# 03. Data Join — Unified Training Set

**목적**: PV(KOEN) + GK-2A v2 + ASOS를 (site, datetime_kst) 기준으로 join하고 학습용 통합 테이블 생성.

**산출물**: `data/processed/training_set.parquet` — 모델 학습 입력

**스키마**:
```
키:        datetime_kst, site, hour, month, doy
타깃:      gen_kwh, cf  (capacity factor)
GHI:       dsr_mean, kt (clearsky index), zenith_center
기상:      ta, hm, ws, dc10Tca, rn  (ASOS, 사이트별 매핑)
메타:      site_capacity_kw, lat, lon, dsr_n_valid
```

**사이트 → ASOS 매핑** (가장 가까운 관측소):
- 경상대 → 진주
- 고흥만수상 → 고흥
- 광양항세방 → 광양시
- 구미 → 구미
- 삼천포 → 진주
- 영흥 → 인천
- 예천 → 안동
- 창원 → 창원

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import pvlib
import warnings

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
warnings.filterwarnings('ignore', message='.*Length of header.*')

# data_strategy.md §2.5 확정 11 호기 → 8 사이트
KEEP_UNITS = [
    ('경상대태양광',     '1',  905, '경상대'),
    ('고흥만 수상태양광', '1', 63481, '고흥만수상'),
    ('광양항세방태양광',  '1', 2993, '광양항세방'),
    ('구미태양광',       '1',  992, '구미'),
    ('삼천포태양광',     '2',  990, '삼천포'),
    ('삼천포태양광',     '3',  350, '삼천포'),
    ('영흥태양광',       '1', 1000, '영흥'),
    ('영흥태양광',       '2',  993, '영흥'),
    ('영흥태양광#5',     '1', 3500, '영흥'),
    ('예천태양광',       '1', 2000, '예천'),
    ('두산엔진MG태양광',  '1',   77, '창원'),
]
UNIT_KEY = {(u, h): (cap, site) for u, h, cap, site in KEEP_UNITS}

SITE_COORDS = {
    '경상대':    (35.18, 128.10),
    '고흥만수상': (34.57, 127.30),
    '광양항세방': (34.93, 127.71),
    '구미':      (36.13, 128.34),
    '삼천포':    (34.95, 128.07),
    '영흥':      (37.26, 126.46),
    '예천':      (36.65, 128.46),
    '창원':      (35.21, 128.58),
}

# 사이트 → ASOS 관측소
SITE_ASOS = {
    '경상대':    '진주',
    '고흥만수상': '고흥',
    '광양항세방': '광양시',
    '구미':      '구미',
    '삼천포':    '진주',
    '영흥':      '인천',
    '예천':      '안동',
    '창원':      '창원',
}

print(f'대상: {len(KEEP_UNITS)} 호기 / {len(SITE_COORDS)} 사이트')
print(f'사이트 ↔ ASOS 매핑: {SITE_ASOS}')

## 1. PV (KOEN) — 사이트 단위 합산

In [ ]:
def load_pv():
    files = sorted(Path('../../data/solar_hourly').glob('solar_hourly_*.csv'))
    dfs = []
    for f in files:
        d = pd.read_csv(f, index_col=False)
        d.columns = [c.strip() for c in d.columns]
        d['발전구분'] = d['발전구분'].astype(str).str.strip()
        d['호기'] = d['호기'].astype(str).str.strip()
        d['일자_str'] = d['일자'].astype(str).str.strip()
        d = d[d['일자_str'].str.match(r'^\d{4}-\d{2}-\d{2}')].copy()
        d['일자'] = pd.to_datetime(d['일자_str'])
        hour_cols = [f'{h}시 발전량(KWh)' for h in range(1, 25)]
        long = d.melt(id_vars=['발전구분','호기','일자'], value_vars=hour_cols,
                      var_name='hl', value_name='gen_kwh')
        long['hour'] = long['hl'].str.extract(r'(\d+)시').astype(int)
        long['datetime_kst'] = long['일자'] + pd.to_timedelta(long['hour'], unit='h')
        long['gen_kwh'] = pd.to_numeric(long['gen_kwh'], errors='coerce')
        long['key'] = list(zip(long['발전구분'], long['호기']))
        long = long[long['key'].isin(UNIT_KEY.keys())].copy()
        long['site_capacity_kw'] = long['key'].map(lambda k: UNIT_KEY[k][0])
        long['site'] = long['key'].map(lambda k: UNIT_KEY[k][1])
        dfs.append(long[['site','datetime_kst','gen_kwh','site_capacity_kw']])
    pv = pd.concat(dfs, ignore_index=True)
    site_pv = (pv.groupby(['site','datetime_kst'], as_index=False)
                 .agg(gen_kwh=('gen_kwh','sum'),
                      site_capacity_kw=('site_capacity_kw','sum')))
    site_pv['cf'] = site_pv['gen_kwh'] / site_pv['site_capacity_kw']
    return site_pv

pv = load_pv()
print(f'PV 행: {len(pv):,}')
print(f'기간: {pv.datetime_kst.min()} ~ {pv.datetime_kst.max()}')
print(f'사이트: {sorted(pv.site.unique())}')

## 2. GK-2A v2 (위성 일사 + zenith)

In [ ]:
gk_files = sorted(Path('../../data/gk2a_v2').glob('*.csv'))
gk = pd.concat([pd.read_csv(f, parse_dates=['datetime_kst']) for f in gk_files], ignore_index=True)
gk = gk[gk.site.isin(SITE_COORDS.keys())][['datetime_kst','site','dsr_mean','dsr_n_valid','zenith_center','lat','lon']].copy()
print(f'GK-2A v2 행: {len(gk):,}, 사이트: {gk.site.nunique()}')

## 3. ASOS — 사이트별 매핑된 기상

In [ ]:
as_files = sorted(Path('../../data/asos_hourly').glob('asos_hourly_*.csv'))
asos = pd.concat([pd.read_csv(f, parse_dates=['tm']) for f in as_files], ignore_index=True)
asos = asos[asos['stnNm'].isin(SITE_ASOS.values())].copy()
asos = asos.rename(columns={'tm':'datetime_kst', 'stnNm':'asos_stn'})

# 변수 선정 — 학습 후보
asos = asos[['datetime_kst', 'asos_stn', 'ta', 'hm', 'ws', 'wd', 'rn', 'pa', 'ps', 'dc10Tca']].copy()
print(f'ASOS 행: {len(asos):,}, 관측소: {sorted(asos.asos_stn.unique())}')
print()
print('--- ASOS 변수별 결측률 (사용 8 관측소) ---')
for col in ['ta','hm','ws','wd','rn','pa','ps','dc10Tca']:
    miss = asos[col].isna().mean() * 100
    print(f'  {col}: {miss:.2f}%')

## 4. Clearsky GHI 계산 (pvlib) → kt index

**kt = dsr_mean / clearsky_ghi** — 사이트 무관 cloudiness 신호. 모델의 가장 강한 derived feature 중 하나.

사용: pvlib `Location.get_clearsky(model='ineichen')` — Linke turbidity 기반 표준 모델.

In [ ]:
def add_clearsky(df, site_coords):
    """각 row의 datetime + site로 clearsky GHI 계산 후 kt 추가."""
    out = []
    for site, group in df.groupby('site', sort=False):
        lat, lon = site_coords[site]
        loc = pvlib.location.Location(lat, lon, tz='Asia/Seoul')
        # 시간 중심 (라벨 - 30min, gk2a v2와 동일 컨벤션)
        time_center = pd.DatetimeIndex(group['datetime_kst']) - pd.Timedelta(minutes=30)
        time_center = time_center.tz_localize('Asia/Seoul')
        cs = loc.get_clearsky(time_center, model='ineichen')
        g = group.copy()
        g['clearsky_ghi'] = cs['ghi'].values
        out.append(g)
    out = pd.concat(out, ignore_index=True)
    # kt 계산: clearsky가 너무 작으면 (dawn/dusk 또는 야간) 무한대 방지
    out['kt'] = np.where(out['clearsky_ghi'] > 50, out['dsr_mean'] / out['clearsky_ghi'], np.nan)
    out['kt'] = out['kt'].clip(lower=0, upper=1.5)   # super-clearsky outlier 클립
    return out

gk_with_kt = add_clearsky(gk, SITE_COORDS)
print('=== kt 통계 (clearsky_ghi > 50 W/m² 기준) ===')
print(gk_with_kt['kt'].describe().round(3))
print()
print('=== 사이트별 kt 평균 (cloudiness 비교) ===')
print(gk_with_kt.groupby('site')['kt'].agg(['mean','std','count']).round(3))

→ **모델 함의**: kt 평균이 사이트별 ~0.5 근처면 평균 cloudiness 비슷. 큰 격차(±0.1+) 있으면 사이트별 cloud climatology 차이 존재.

## 5. Cyclical 시간 인코딩

In [ ]:
def add_time_features(df):
    df = df.copy()
    df['hour'] = df['datetime_kst'].dt.hour
    df['month'] = df['datetime_kst'].dt.month
    df['doy'] = df['datetime_kst'].dt.dayofyear
    # cyclical encoding (TFT 같은 시퀀스 모델에 도움, LGBM은 raw도 OK)
    df['hour_sin'] = np.sin(2*np.pi*df['hour']/24)
    df['hour_cos'] = np.cos(2*np.pi*df['hour']/24)
    df['month_sin'] = np.sin(2*np.pi*df['month']/12)
    df['month_cos'] = np.cos(2*np.pi*df['month']/12)
    df['doy_sin'] = np.sin(2*np.pi*df['doy']/366)
    df['doy_cos'] = np.cos(2*np.pi*df['doy']/366)
    return df

## 6. Join all → unified training set

In [ ]:
# 사이트 → ASOS 매핑 broadcasting
asos_renamed = asos.copy()
asos_renamed['site'] = asos_renamed['asos_stn'].map({v: k for k, v in SITE_ASOS.items()})
# 같은 ASOS 관측소(예: 진주)가 여러 사이트(경상대, 삼천포)에 매핑 → 중복 row
site_asos_pairs = pd.DataFrame([
    {'site': site, 'asos_stn': stn} for site, stn in SITE_ASOS.items()
])
asos_per_site = asos.merge(site_asos_pairs, on='asos_stn')   # 진주 → 경상대 + 삼천포 둘 다로 broadcast

# 1차 join: PV ⋈ GK-2A
df = pv.merge(gk_with_kt, on=['datetime_kst', 'site'], how='inner')
# 2차 join: + ASOS
df = df.merge(asos_per_site[['datetime_kst','site','ta','hm','ws','wd','rn','dc10Tca','pa','ps']],
              on=['datetime_kst','site'], how='left')
# 3차: 시간 feature
df = add_time_features(df)

print(f'Unified training set: {len(df):,} 행, {df.site.nunique()} 사이트')
print(f'기간: {df.datetime_kst.min()} ~ {df.datetime_kst.max()}')
print()
print('--- 컬럼 ---')
print(df.columns.tolist())

## 7. 품질 진단

In [ ]:
print('=== 결측률 (학습 candidate 변수) ===')
feat_cols = ['dsr_mean','kt','zenith_center','ta','hm','ws','wd','rn','dc10Tca','pa','ps']
for c in feat_cols:
    miss = df[c].isna().mean() * 100
    print(f'  {c:<20} {miss:>6.2f}%')

print()
print('=== 사이트별 행 수 ===')
print(df.groupby('site').size().to_string())

print()
print('=== Target (cf) 분포 ===')
print(df['cf'].describe().round(3))

## 8. 저장

In [ ]:
out_dir = Path('../../data/processed')
out_dir.mkdir(exist_ok=True)
out_path = out_dir / 'training_set.parquet'
df.to_parquet(out_path, index=False)
print(f'저장: {out_path}')
print(f'크기: {out_path.stat().st_size / 1024**2:.1f} MB')

# CSV도 같이 (디버깅용)
df.to_csv(out_dir / 'training_set.csv', index=False)
print(f'저장: {out_dir / "training_set.csv"}')

## 9. 다음 단계

- **노트북 04**: NGBoost baseline 학습 (CPU, 분 단위)
- **노트북 05**: TFT 학습 (GPU, 1~3시간)

**학습 변수 결정 (앞서 합의)**:
```python
features_core = [
    'dsr_mean', 'kt', 'zenith_center',  # 일사·천문
    'ta', 'hm', 'ws', 'dc10Tca',         # 기상
    'hour', 'month',                      # 시간 (또는 cyclical)
    'site_id',                            # categorical
]
y = 'cf'
```